# `ClassBase`

`nematics3d.core.class_base.ClassBase` is the common object foundation used by many Nematics3D classes.

Users normally do **not** construct `ClassBase` directly. Instead, they work with concrete classes that inherit a shared object protocol from it. This tutorial explains that shared protocol: how to identify an unfamiliar object, inspect what it contains, understand attribute and method prefixes, see what can be modified, and inspect relations to other objects.


## What `ClassBase` is for

Different Nematics3D classes can represent very different things, but they often need the same basic object behavior.

A `ClassBase` descendant can expose a stable name, documented readable attributes, controlled assignment, and relations to other Nematics3D objects. The scientific or numerical meaning still belongs to the concrete subclass; `ClassBase` supplies the common object language.

A useful mental model is:

```text
ClassBase
    └── common Nematics3D object behavior
            ├── identity
            ├── attribute discovery
            ├── controlled assignment
            └── object relations

Concrete subclass
    └── the actual scientific, numerical, or organizational behavior
```


## Setup

This tutorial uses `SmoothedLine` as a concrete example. A `SmoothedLine` stores a polyline and produces a smoothed version of it, so its basic role is easy to understand without any liquid-crystal or other domain-specific background.

`SmoothedLine` actually inherits through `HostBase`, which itself extends `ClassBase`. We will ignore the extra host/opts machinery here and use the object only to learn the common `ClassBase` interface. That host layer is introduced separately in the `HostBase` tutorial.


In [ ]:
import numpy as np
import nematics3d as n3d

t = np.linspace(0.0, 4.0 * np.pi, 101)
coords = np.column_stack([
    t,
    np.sin(t) + 0.08 * np.sin(9.0 * t),
    0.25 * np.cos(0.5 * t),
])

line = n3d.SmoothedLine(
    coords,
    name="example line",
    window_length=11,
)

line


## Start with an unfamiliar object

When you receive an unfamiliar Nematics3D object, the first useful question is:

> What kind of object is this?

Use:

```python
line.show_doc()
```

`show_doc()` displays the class docstring of the **concrete class of the current object**. Here it therefore shows the `SmoothedLine` docstring, not the `ClassBase` docstring.


In [ ]:
line.show_doc()


`show_doc()` answers **what this object represents and what it is for** before you start inspecting individual fields.


## What can I read?

Use:

```python
line.show_readable_attrs()
```

to list the registered user-readable attributes and their descriptions.


In [ ]:
line.show_readable_attrs()


If one field is unfamiliar, inspect only that field:

```python
line.show_attr_doc("raw_coords")
```


In [ ]:
line.show_attr_doc("raw_coords")


## Prefix conventions

Nematics3D uses a small set of prefixes as part of its object vocabulary. They tell you what role an attribute or method plays before you know the details of the concrete class.

The prefixes are useful for reading code, but they also provide a practical way to **discover an unfamiliar object's API**.


### Attribute prefixes

| Prefix or form | Meaning |
| --- | --- |
| `raw_...` | Canonical stored public input or base data. A shorter alias without `raw_` is normally available for reading. |
| `state_...` | Writable runtime state describing the current state of the object. |
| `default_...` | A managed default-layer input. |
| `calc_...` | A computed readable result. Normally read-only from the public interface. |
| `entity_...` | A computed or generated object-valued result. Normally read-only. |
| `impl_...` | Internal implementation state. Ordinary users should normally ignore it. |
| no prefix, relation | A semantic link to another object, such as `owner` or `registry`. |
| no prefix, property | A normal Python property; its precise role is documented by the concrete class. |

For `SmoothedLine`, examples include `raw_coords`, `state_is_window_warning`, `calc_result`, `calc_status`, `entity_tck`, and internal `impl_...` fields.


### **Use prefixes with autocomplete**

**One major practical advantage of the prefix convention is autocomplete-based discovery.**

Suppose you know that an object stores a calculated quantity, but you do not remember its exact name. Type:

```python
line.calc_
```

and use your editor or notebook's autocomplete (for example, press **Tab**) to browse the calculated quantities available on that object.

For a `SmoothedLine`, this immediately narrows the search to names such as `calc_coords`, `calc_is_smoothed`, `calc_num_init`, `calc_num_out`, `calc_result`, and `calc_status`.

The same idea works for the other categories:

```python
line.raw_       # What base data does this object store?
line.state_     # What runtime state does it expose?
line.calc_      # What has it calculated?
line.entity_    # What object-valued results has it created?
```

You therefore do not need to memorize every attribute name before using an unfamiliar Nematics3D object.


### `raw_` and the public alias

`raw_` is for canonical stored input or base data, rather than a value produced by the object's calculation.

For `SmoothedLine`, the clearest example is the original input polyline:

```python
line.raw_coords
```

`ClassBase` normally also provides a shorter readable alias without the `raw_` prefix:


In [ ]:
line.raw_coords, line.coords


Both names refer to the same underlying input coordinates. The explicit `raw_coords` form is useful when you want to emphasize the field's role as canonical stored input; `coords` is convenient in ordinary analysis code.

This contrasts naturally with calculated fields such as `line.calc_result` or `line.calc_status`, which describe outputs of the smoothing pipeline rather than the original input.


### Method prefixes

Public methods also follow a small prefix convention:

| Prefix | Meaning |
| --- | --- |
| `act_...` | Perform an action or operation on/through the object. |
| `show_...` | Show, inspect, or explain information about the object. |

This gives the same autocomplete advantage:

```python
line.act_       # What can this object do?
line.show_      # What can this object show or explain?
```

For example, `line.act_commit(...)` performs a host update, while `line.show_readable_attrs()` and `line.show_relations()` inspect the object.

The details of `act_commit(...)` belong to `HostBase` and are discussed later; here the important point is the common naming vocabulary and how it helps you discover methods.


## What can I modify?

A `ClassBase` object is not an unrestricted Python attribute bag. A concrete class can distinguish writable inputs from computed outputs, protected fields, and fixed core data.

Instead of guessing, use:

```python
line.show_modifiable_attrs()
```


In [ ]:
line.show_modifiable_attrs()


This list is instance-aware. A field can exist and be readable without being modifiable on the current object.

When public assignment is allowed, `ClassBase` routes it through the object's validation and protection rules rather than blindly storing an arbitrary value.


## Object identity

Every `ClassBase` object has a readable name through `name`, backed by `raw_name`.


In [ ]:
line.raw_name, line.name


The public alias can also be used for renaming:

```python
line.name = "new name"
```

If the object participates in a registry or another naming constraint, the object system can apply those rules during assignment.


## Relations between objects

`ClassBase` also provides a common way to represent semantic links between objects. Typical examples are `owner` and `registry`.

Use:

```python
line.show_relations()
```

to display currently bound relations and their targets.


In [ ]:
line.show_relations()


For objects embedded in a larger object graph, use:

```python
line.show_relation_tree(depth=2)
```

to follow declared relations recursively.


## A practical inspection workflow

When an unfamiliar Nematics3D object appears in a notebook, a useful default workflow is:

```python
obj.show_doc()                  # What is this object?
obj.show_readable_attrs()       # What can I read?
obj.show_attr_doc("...")        # What does one field mean?
obj.show_modifiable_attrs()     # What can I change?
obj.show_relations()            # What other objects is it connected to?
```

When you only remember the *kind* of thing you want, autocomplete can be even faster:

```python
obj.raw_
obj.state_
obj.calc_
obj.entity_

obj.act_
obj.show_
```

You will not always need every step. The point is that the same questions and the same small vocabulary work across many otherwise unrelated Nematics3D classes.


## Where `HostBase` enters

`SmoothedLine` is more than a plain `ClassBase`: it inherits from `HostBase`, which adds a paired options object and a commit-style update pipeline.

That extra layer is intentionally not explained here. The conceptual progression is:

```text
ClassBase
    ↓
common object language and inspection

HostBase
    ↓
opts + commit/update machinery

SmoothedLine
    ↓
actual line-smoothing behavior
```

Using the same `SmoothedLine` object in the next tutorial lets you learn the additional `HostBase` layer without changing examples.


## What `ClassBase` does not do

`ClassBase` does not perform the scientific or numerical calculation of its subclasses. It does not know how to smooth a line, fit a plane, diagonalize a tensor, or render a figure.

It provides the common object protocol underneath those domain-specific classes. Learning `ClassBase` is therefore learning the **common language of Nematics3D objects**, not learning one particular algorithm.


# For developers

The user-facing behavior above is implemented through the class-level attribute schema and per-instance relation and assignment state.

Developers defining a new `ClassBase` descendant should follow the established attribute kinds and prefix conventions so the new class participates in the same inspection and autocomplete-driven discovery workflow. Ordinary users do not need to understand that internal machinery in order to use the object.
